# IEEE-CIS Fraud Detection: Data Profiling & Schema Validation (No Charts)

This notebook performs pure **numerical data profiling**, validation checks, and architectural pruning for the Data Engineering pipeline without plotting dependencies.
Visual charts and business findings are documented in [`docs/data_exploration.md`](../docs/data_exploration.md).


---
## 1. Environment & Setup
Load standard data processing libraries.


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Path to raw dataset
DATA_DIR = Path("../data/raw/ieee-fraud-detection")
print(f"Data Directory: {DATA_DIR.resolve()}")


Data Directory: /mnt/d/4.DE-zoomcamp/projects/fraud-risk-control-analytics/data/raw/ieee-fraud-detection


---
## 2. Data Ingestion & Memory Profiling
Inspect table dimensions, data types, and initial memory footprints.


In [11]:
df_trans = pd.read_csv(DATA_DIR / "train_transaction.csv")
df_id = pd.read_csv(DATA_DIR / "train_identity.csv")

print("=== Dimensions ===")
print(f"Transactions: {df_trans.shape[0]:,} rows x {df_trans.shape[1]} columns")
print(f"Identities:   {df_id.shape[0]:,} rows x {df_id.shape[1]} columns")

print("=== Memory Usage ===")
print(f"Transactions: {df_trans.memory_usage(deep=True).sum() / (1024**2):.1f} MB")
print(f"Identities:   {df_id.memory_usage(deep=True).sum() / (1024**2):.1f} MB")


=== Dimensions ===
Transactions: 590,540 rows x 394 columns
Identities:   144,233 rows x 41 columns
=== Memory Usage ===
Transactions: 1791.7 MB
Identities:   56.5 MB


---
## 3. Ground Truth Metrics: Target & Amount Distributions
Quantify target class imbalance and transaction amount percentiles.


In [13]:
# Target Class Imbalance (isFraud)
fraud_summary = pd.DataFrame({
    'count': df_trans['isFraud'].value_counts(),
    'percentage': df_trans['isFraud'].value_counts(normalize=True) * 100
})
print("=== Target Distribution (isFraud) ===")
print(fraud_summary)

# Transaction Amount Percentiles
print("=== Transaction Amount ($ USD) Percentiles ===")
print(df_trans['TransactionAmt'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))


=== Target Distribution (isFraud) ===
          count  percentage
isFraud                    
0        569877   96.500999
1         20663    3.499001
=== Transaction Amount ($ USD) Percentiles ===
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
90%         275.293000
95%         445.000000
99%        1104.000000
max       31937.391000
Name: TransactionAmt, dtype: float64


---
## 4. Key Integrity & Risk Segmentation
Check key uniqueness, matching coverage, and fraud rates across identity presence and device types.


In [17]:
# Key Uniqueness & Match Rate
print("TransactionID is unique in Trans:    ", df_trans['TransactionID'].is_unique)
print("TransactionID is unique in Identity: ", df_id['TransactionID'].is_unique)

match_pct = df_trans['TransactionID'].isin(df_id['TransactionID']).mean() * 100
print(f"Matching Identity Coverage: {match_pct:.2f}% ({df_trans['TransactionID'].isin(df_id['TransactionID']).sum():,} transactions)")

# Risk Segmentation: Identity Presence & Device Type
df_merged = df_trans[['TransactionID', 'isFraud']].merge(
    df_id[['TransactionID', 'DeviceType']], on='TransactionID', how='left'
)
df_merged['has_identity'] = df_merged['DeviceType'].notna()

print("=== Fraud Rate: With vs Without Identity ===")
id_risk = df_merged.groupby('has_identity')['isFraud'].agg(['count', 'mean'])
id_risk['fraud_rate_pct'] = id_risk['mean'] * 100
print(id_risk[['count', 'fraud_rate_pct']])

print("=== Fraud Rate: By Device Type ===")
dev_risk = df_merged.dropna(subset=['DeviceType']).groupby('DeviceType')['isFraud'].agg(['count', 'mean'])
dev_risk['fraud_rate_pct'] = dev_risk['mean'] * 100
print(dev_risk[['count', 'fraud_rate_pct']])

TransactionID is unique in Trans:     True
TransactionID is unique in Identity:  True
Matching Identity Coverage: 24.42% (144,233 transactions)
=== Fraud Rate: With vs Without Identity ===
               count  fraud_rate_pct
has_identity                        
False         449730        2.101705
True          140810        7.961792
=== Fraud Rate: By Device Type ===
            count  fraud_rate_pct
DeviceType                       
desktop     85165        6.521458
mobile      55645       10.166232


---
## 5. Data Warehouse Architecture: Column Pruning & Normalization
Prune out 339 sparse `V` columns and validate the lean schema (~20-22 columns) for downstream staging & marts.


In [18]:
# Selected core columns for DW
SELECTED_TRANSACTION_COLS = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD',
    'card1', 'card2', 'card4', 'card6',
    'addr1', 'addr2', 'dist1',
    'P_emaildomain', 'R_emaildomain'
]

SELECTED_IDENTITY_COLS = [
    'TransactionID', 'DeviceType', 'DeviceInfo', 'id_30', 'id_31'
]

df_trans_lean = df_trans[SELECTED_TRANSACTION_COLS]
df_id_lean = df_id[SELECTED_IDENTITY_COLS]

print("=== Column Pruning Summary ===")
print(f"Transactions: {df_trans.shape[1]} cols -> {df_trans_lean.shape[1]} cols")
print(f"Identities:   {df_id.shape[1]} cols -> {df_id_lean.shape[1]} cols")

raw_mem = df_trans.memory_usage().sum() / (1024**2)
lean_mem = df_trans_lean.memory_usage().sum() / (1024**2)
print(f"Memory reduction: {raw_mem:.1f} MB -> {lean_mem:.1f} MB (~{(1 - lean_mem/raw_mem)*100:.1f}% reduction)")

# Column name standardization check (replace hyphens with underscores)
def standardize_cols(df):
    return df.rename(columns=lambda c: c.replace("-", "_").lower())

print("=== Standardized Column Names Sample ===")
print(list(standardize_cols(df_id_lean).columns))


=== Column Pruning Summary ===
Transactions: 394 cols -> 14 cols
Identities:   41 cols -> 5 cols
Memory reduction: 1791.7 MB -> 76.1 MB (~95.8% reduction)
=== Standardized Column Names Sample ===
['transactionid', 'devicetype', 'deviceinfo', 'id_30', 'id_31']
